In [ ]:
from committor_control import *

import pandas as pd
import numpy as np
import os, sys, re
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from pathlib import Path

plt.rcParams['xtick.labelsize'] = 14  # font size
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams["font.family"] = "serif"
mpl.rcParams['pdf.fonttype'] = 42  # Use TrueType fonts (editable in Illustrator)
mpl.rcParams['ps.fonttype'] = 42   # Same for EPS
mpl.rcParams['svg.fonttype'] = 'none'  # Keep text as text in SVG
mpl.rcParams["mathtext.fontset"] =  "cm"

In [ ]:

if __name__ == "__main__":
    dt = 0.01
    num_trajs = 1000

    sys_dw   = make_double_well_curved(lambda_u=2.0, lambda_s=8.0, c=0.5)
    sys_lin  = make_linear_system(lambda_u=2.0, lambda_s=-8.0)
    sys_pend = make_pendulum(g=1.5, length=1.0, damping=0.0)

    use_whitened = True

    systems = {
        "linear_2d":   (sys_lin,  1.0, np.inf, 10, 0.01),
        "double_well": (sys_dw,   0.5, 0.5,    10, 0.01),
        "pendulum":    (sys_pend, 1.0, 1.0,    30, 0.01),
    }

    # ══════════════════════════════════════════════════════════════════════
    # FIGURE 1: Single trajectory examples (all 3 systems)
    #   Autonomous / Data-enabled (open-loop) / Closed-loop (adaptive)
    # ══════════════════════════════════════════════════════════════════════
    print("\n" + "#"*80 + "\n FIGURE 1: Trajectory examples\n" + "#"*80)
    traj_methods = ["Autonomous", "CCA (Whitened)", "CCA (Adaptive)"]
    for tag, (sys, a, r_neigh, tau, D_eval) in systems.items():
        run_evaluation_suite(sys, a=a, r_neigh=r_neigh, D_eval=D_eval, dt=dt,
                             num_trajs=100, plot_trajs=3, spread_s=0.75, tau=tau,
                             methods_to_run=traj_methods, use_whitened=use_whitened,
                             file_suffix="trajs")

    # ══════════════════════════════════════════════════════════════════════
    # FIGURE 2: Linear system diagnostics
    #   2a. Parameter convergence (stacked and combined)
    #   2b. Noise sweep (success + energy vs D)
    #   2c. Dimensionality & lambda scaling
    #   2d. Combined scaling + energy
    # ══════════════════════════════════════════════════════════════════════
    print("\n" + "#"*80 + "\n FIGURE 2: Linear diagnostics\n" + "#"*80)
    sys, a, r_neigh, tau, D_eval = systems["linear_2d"]

    # 2a. Convergence
    conv_csv = "figures/linear_2d_convergence.csv"
    generate_convergence_data(sys, a=a, D_train=D_eval, dt=dt, T_train=500,
                              r_neigh=r_neigh, num_trajs=num_trajs, tau=tau,
                              use_whitened=use_whitened, csv_filepath=conv_csv)
    plot_convergence_from_data(csv_filepath=conv_csv, sys_lambda_true=sys.lambda_u_true,
                               sys_D_true=D_eval, filepath="figures/linear_2d_convergence_stacked.pdf",
                               layout="stacked", label_fontsize=13, tick_fontsize=12, legend_fontsize=10)
    plot_convergence_from_data(csv_filepath=conv_csv, sys_lambda_true=sys.lambda_u_true,
                               sys_D_true=D_eval, filepath="figures/linear_2d_convergence_combined.pdf",
                               layout="combined", label_fontsize=13, tick_fontsize=12, legend_fontsize=10)

    # 2b. Noise sweep (D)
    energy_csv = "figures/linear_2d_noise_sweep.csv"
    simulate_and_save_noise_energy_sweep(sys, a=a, dt=dt, r_neigh=r_neigh,
                                         num_trajs=num_trajs, T_sim=2000, tau=tau,
                                         use_whitened=use_whitened, csv_filepath=energy_csv)
    plot_success_and_energy_vs_noise(csv_filepath=energy_csv, sys_name=sys.name,
                                     filepath="figures/linear_2d_noise_sweep.pdf",
                                     layout="stacked", include_adaptive=False)

    # 2c. Dimensionality & lambda scaling
    run_dimensionality_scaling_experiment(dt=dt, num_trajs=num_trajs, T_sim=5000, use_whitened=use_whitened)

    # 2d. Combined scaling + energy
    plot_scaling_and_energy(dim_csv_filepath="figures/dimensionality_scaling_stats.csv",
                            energy_csv_filepath=energy_csv,
                            include_adaptive=False, line_alpha=0.7,
                            filepath="figures/linear_2d_combined_scaling_energy.pdf")

    # ══════════════════════════════════════════════════════════════════════
    # FIGURE 3: Nonlinear systems — noise sweep (D) only
    # ══════════════════════════════════════════════════════════════════════
    print("\n" + "#"*80 + "\n FIGURE 3: Nonlinear noise sweeps\n" + "#"*80)
    for tag in ["double_well", "pendulum"]:
        sys, a, r_neigh, tau, D_eval = systems[tag]
        csv_path = f"figures/{tag}_noise_sweep.csv"
        simulate_and_save_noise_energy_sweep(sys, a=a, dt=dt, r_neigh=r_neigh,
                                              num_trajs=num_trajs, T_sim=2000, tau=tau,
                                              use_whitened=use_whitened, csv_filepath=csv_path)
        plot_success_and_energy_vs_noise(csv_filepath=csv_path, sys_name=sys.name,
                                          filepath=f"figures/{tag}_noise_sweep.pdf",
                                          layout="stacked", include_adaptive=False)

### CCA explanation plot

In [ ]:
"""
Poster figure: Before/after temporal differencing scatter plot.
Shows that differencing removes the unknown fixed point x*.
Now includes the CCA-estimated \alpha_0 vector.
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

# ── Style ──
plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",  
    "font.size": 12,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "axes.linewidth": 1.2,
    "axes.edgecolor": "#333333",
    "axes.facecolor": "#fafafa",
    "figure.facecolor": "white",
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
})

stroke = pe.withStroke(linewidth=3, foreground="white")
fontsize = 18

# ── System setup ──
lambda_u, lambda_s = 2.0, -8.0
theta = 0.4
Q = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
J = Q @ np.diag([lambda_u, lambda_s]) @ np.linalg.inv(Q)
J = np.array([[ 0.78830835,  0.16234692],
              [ 2.3318579 , -0.78830835]])
x_star = np.array([1.3, -0.7])

# Eigenvectors
eigvals_l, V_left = np.linalg.eig(J.T)
w_u = V_left[:, np.argmax(eigvals_l.real)].real
w_u /= np.linalg.norm(w_u)

eigvals_r, V_right = np.linalg.eig(J)
v_u = V_right[:, np.argmax(eigvals_r.real)].real; v_u /= np.linalg.norm(v_u)
v_s = V_right[:, np.argmin(eigvals_r.real)].real; v_s /= np.linalg.norm(v_s)

# ── Simulate ──
dt, D, T, tau = 0.01, 0.01, 2000, 1
rng = np.random.default_rng(42)
noise_std = np.sqrt(2.0 * D)

x = x_star + rng.uniform(-0.1, 0.1, 2)
X = np.zeros((T, 2))
is_reset = np.zeros(T, dtype=bool)

for t in range(T):
    X[t] = x
    x = x + J @ (x - x_star) * dt + noise_std * rng.standard_normal(2) * np.sqrt(dt)
    if np.linalg.norm(x - x_star) > 0.5:
        x = x_star + rng.uniform(-0.05, 0.05, 2)
        is_reset[t] = True

# Valid differences only for plotting
valid = ~is_reset[tau:] & ~is_reset[:-tau]
P = (X[tau:] - X[:-tau])[valid]


# ── CCA Calculation for \alpha_0 ──
P_all = X[tau:] - X[:-tau]

# For CCA, we need valid consecutive pairs (p_t, f_t) where f_t = p_{t+1}
valid_cca = valid[:-1] & valid[1:] 
p_t = P_all[:-1][valid_cca]
f_t = P_all[1:][valid_cca]

N_cca = len(p_t)
C_pp = (p_t.T @ p_t) / N_cca
C_ff = (f_t.T @ f_t) / N_cca
C_pf = (p_t.T @ f_t) / N_cca
C_fp = (f_t.T @ p_t) / N_cca

# Solve eigenvalue problem: C_pp^-1 C_pf C_ff^-1 C_fp \alpha = \rho^2 \alpha
M_cca = np.linalg.inv(C_pp) @ C_pf @ np.linalg.inv(C_ff) @ C_fp
eigvals_cca, V_cca = np.linalg.eig(M_cca)

# Leading eigenvector is \alpha_0
alpha_0 = V_cca[:, np.argmax(eigvals_cca.real)].real
alpha_0 /= np.linalg.norm(alpha_0) # L2 normalization for plotting alongside w_u

# Align sign with w_u for easy visual comparison
if np.dot(alpha_0, w_u) < 0:
    alpha_0 = -alpha_0


# ── Figure ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.2),
                               gridspec_kw={"wspace": 0.32})

# ═══ Left panel: raw x_t ═══
ax1.scatter(X[:, 0], X[:, 1], s=5, alpha=0.2, color="#888888",
            edgecolors="none", zorder=3)

# Fixed point
ax1.scatter(*x_star, s=140, color="#000000", marker="X", linewidths=0,
            zorder=6, edgecolors="white")
ax1.text(x_star[0] - 0.02, x_star[1] + 0.04, r"$x^\ast$",
         fontsize=fontsize, color="#000000", fontweight="bold",
         path_effects=[stroke], ha="right", zorder=7)

# Right eigenvectors — short arrows
arrow_l = 0.15
for v, lab, col, ha, va, dx, dy in [
    (v_u, r"$v_u$", "#e67e22", "left",  "center", 0.02, 0.0),
    (v_s, r"$v_s$",   "#8e44ad", "center","bottom", 0.0,  0.02),
]:
    tip = x_star + arrow_l * v
    ax1.annotate("", xy=tip, xytext=x_star,
                 arrowprops=dict(arrowstyle="-|>", color=col, lw=2.2,
                                 mutation_scale=13), zorder=5)
    ax1.text(tip[0] + dx, tip[1] + dy, lab,
             color=col, fontsize=fontsize, fontweight="bold", ha=ha, va=va,
             path_effects=[stroke], zorder=8)

ax1.set_title(r"Raw observations $x_t$", fontsize=fontsize, pad=12)
ax1.set_xlabel(r"$x_1$", fontsize=fontsize)
ax1.set_ylabel(r"$x_2$", fontsize=fontsize)
ax1.set_aspect("equal")
ax1.set_xticks([])
ax1.set_yticks([])

# ═══ Right panel: temporal differences ═══
ax2.scatter(P[:, 0], P[:, 1], s=5, alpha=0.25, color="#27ae60",
            edgecolors="none", zorder=3)

# Origin
ax2.scatter(0, 0, s=140, color="black", marker="+", linewidths=2.5, zorder=6)

# Eigenvectors — scale to data spread
arrow_r = np.percentile(np.linalg.norm(P, axis=1), 90) * 1.0
for v, lab, col, ha, va, dx, dy in [
    (v_u, r"$v_u$", "#e67e22", "left",   "center", 0.002, 0.0),
    (v_s, r"$v_s$",   "#8e44ad", "center", "bottom", 0.0,   0.003),
]:
    tip = arrow_r * v
    ax2.annotate("", xy=tip, xytext=[0, 0],
                 arrowprops=dict(arrowstyle="-|>", color=col, lw=2.2,
                                 mutation_scale=13), zorder=5)
    ax2.text(tip[0] + dx, tip[1] + dy, lab,
             color=col, fontsize=fontsize, fontweight="bold", ha=ha, va=va,
             path_effects=[stroke], zorder=8)

# True Left unstable eigenvector (w_u)
tip_wu = arrow_r * 0.85 * w_u
ax2.annotate("", xy=tip_wu, xytext=[0, 0],
             arrowprops=dict(arrowstyle="-|>", color="#2a2d7c", lw=3.5,
                             mutation_scale=13), zorder=6)
ax2.text(tip_wu[0] - 0.006, tip_wu[1] - 0.004,
         r"$w_u$", color="#2a2d7c", fontsize=fontsize,
         fontweight="bold", ha="left", va="top",
         path_effects=[stroke], zorder=8)

# CCA Estimated eigenvector (\alpha_0)
tip_alpha = arrow_r * 0.85 * alpha_0
ax2.annotate("", xy=tip_alpha, xytext=[0, 0],
             arrowprops=dict(arrowstyle="-|>", color="#e74c3c", lw=2.0, ls="--",
                             mutation_scale=13), zorder=7)
ax2.text(tip_alpha[0] + 0.01, tip_alpha[1] + 0.00,
         r"$\alpha_0$", color="#e74c3c", fontsize=fontsize,
         fontweight="bold", ha="right", va="bottom",
         path_effects=[stroke], zorder=8)

ax2.set_title(r"Temporal differences $p_t = x_t - x_{t-\tau}$", fontsize=fontsize, pad=12)
ax2.set_xlabel(r"$p_1$", fontsize=fontsize)
ax2.set_ylabel(r"$p_2$", fontsize=fontsize)
ax2.set_aspect("equal")
ax2.set_xticks([])
ax2.set_yticks([])

fig.savefig("figures/differencing_scatter.pdf", dpi=300, bbox_inches="tight",
            facecolor="white", edgecolor="none")
print("Saved -> figures/differencing_scatter.pdf")